# TRAIN-001 — PhoWhisper Fine-tune (Colab / Kaggle)

MediVoice VN — `fids/FID-VN-007.md` (v1+v2+v3 DONE). Consolidated notebook: clone -> install -> download VietMed -> (optional pilot audio) -> build manifest -> smoke-test -> full fine-tune -> evaluate (WER) -> download checkpoint -> cleanup.

**Compliance**: `docs/records/DECISIONS.md` 2026-06-11 (PILOT PHASE EXCEPTION #2) + `docs/compliance/RISK_REGISTER.md` R-P03. Uploading pilot audio with patient PII to Colab/Kaggle (Google Cloud, ngoai VN) is TEMPORARY, pilot-only, consent confirmed by Andy. **Delete pilot audio from the runtime/Drive/Dataset immediately after the training run** — only download the model checkpoint (.bin/.safetensors, no audio/PII). VietMed (MIT, public, no PII) has no such restriction.

## 1. Clone repo

In [ ]:
!git clone https://github.com/vietsharescom/MediVoice_VN.git
%cd MediVoice_VN

## 2. Install dependencies

In [ ]:
!pip install -q transformers datasets librosa soundfile accelerate evaluate jiwer

## 3. (Optional) HF_TOKEN

`leduckhai/VietMed` is **NOT gated** — no token required. `HF_TOKEN` is optional and only raises HuggingFace rate limits. Never paste the token directly into a cell or commit it.

In [ ]:
import os

# Colab: Secrets panel (key icon) -> add HF_TOKEN, then uncomment:
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Kaggle: Add-ons -> Secrets -> add HF_TOKEN, then uncomment:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 4. Download VietMed (16h, MIT, no PII) -> `data/vietmed/`

In [ ]:
!python -X utf8 scripts/download_vietmed.py --split all

## 5. (Optional, pilot-only) Upload pilot audio -> `data/audio/pilot/`

Only if 50-100h pilot audio has been recorded **and** consent confirmed (`docs/records/DECISIONS.md` 2026-06-11). Each `*.wav` needs a sibling `*.txt` (plain transcript) or `*.json` (`{"text": ...}`). Upload via the Colab file-upload UI or a temporary Drive mount — do NOT leave it in a shared/public folder.

In [ ]:
from pathlib import Path

PILOT_DIR = Path("data/audio/pilot")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

# Colab upload UI:
# from google.colab import files
# uploaded = files.upload()  # then move uploaded *.wav/*.txt/*.json into PILOT_DIR

print("Pilot audio files found:", len(list(PILOT_DIR.glob("*.wav"))))

## 6. Build manifests -> `data/asr_manifest/`

Writes `ref_voice_manifest.jsonl` (57 ref clips), `vietmed_manifest.jsonl`, `pilot_manifest.jsonl` (if `data/audio/pilot/` has entries), and `combined_manifest.jsonl` (all three).

In [ ]:
!python -X utf8 scripts/build_asr_manifest.py --vietmed --pilot data/audio/pilot --combined

## 7. Smoke test (CPU/GPU, validates pipeline)

In [ ]:
!python -X utf8 scripts/train_asr_phowhisper.py --smoke-test

## 8. Full fine-tune run (GPU)

`--manifest` = `data/asr_manifest/combined_manifest.jsonl` if pilot audio was added in step 5 (recommended for the real TRAIN-001 run), otherwise `data/asr_manifest/vietmed_manifest.jsonl` (VietMed-only, no PII). Output checkpoint -> `models/asr_phowhisper/` (gitignored).

**Do NOT interrupt this cell (no `Ctrl+C` / stop button) once it starts.** The first 3-5 min are dataset `.map()` preprocessing (9207 VietMed examples) — this is normal, not a hang. After that, training runs with `--batch 2 --grad-accum 4` (effective batch 8, fits a 14.5GB T4/P100 — `--batch 8` OOMs). On a T4 with `--fp16`, expect roughly **45-90 min/epoch**, so **2-4.5h total for 3 epochs**. If your session has a shorter time limit (e.g. free Kaggle GPU quota), reduce `--epochs 1` for a first run and increase later.

In [ ]:
MANIFEST = "data/asr_manifest/combined_manifest.jsonl"  # or vietmed_manifest.jsonl if no pilot audio

# --batch is per-device batch size = the actual memory driver (NOT --grad-accum).
# --batch 8 OOMs on a 14.5GB GPU (T4/P100). --batch 2 --grad-accum 4 keeps effective
# batch = 8 while fitting memory; --fp16 also enables gradient_checkpointing for extra headroom.
!python -X utf8 scripts/train_asr_phowhisper.py \
    --manifest {MANIFEST} \
    --epochs 1 --batch 2 --grad-accum 4 --fp16

## 9. Evaluate — WER before vs after fine-tune

`scripts/eval_asr_phowhisper.py` transcribes an eval manifest with the **base** model (`vinai/PhoWhisper-medium`) and the **fine-tuned** checkpoint (`models/asr_phowhisper/`), computes WER via jiwer, and writes JSON results to `data/eval/`.

`data/asr_manifest/ref_voice_manifest.jsonl` (57 real BS-voice clips, held out of training) is the **preferred** eval set, but it is **empty on a fresh Colab/Kaggle clone** — `data/audio/reference_voices/` is gitignored (real BS PII audio, not in the repo). The cell below auto-detects this: if `ref_voice_manifest.jsonl` is empty, it falls back to a 30-sample slice of `data/asr_manifest/vietmed_manifest.jsonl` (built in step 6, no PII) so the eval always has data to run on.

In [ ]:
from pathlib import Path

# ref_voice_manifest.jsonl is empty on Colab/Kaggle (data/audio/reference_voices/ is
# gitignored, 57 real BS clips not present after git clone) -> fall back to a VietMed
# slice (no PII) so the eval always has data to run on.
_ref_manifest = Path("data/asr_manifest/ref_voice_manifest.jsonl")
if _ref_manifest.exists() and _ref_manifest.stat().st_size > 0:
    EVAL_MANIFEST = str(_ref_manifest)
    LIMIT_ARGS = []
else:
    EVAL_MANIFEST = "data/asr_manifest/vietmed_manifest.jsonl"
    LIMIT_ARGS = ["--limit", "30"]

print("EVAL_MANIFEST:", EVAL_MANIFEST, "| extra args:", LIMIT_ARGS or "none")

# Baseline (pre-fine-tune)
!python -X utf8 scripts/eval_asr_phowhisper.py \
    --model vinai/PhoWhisper-medium \
    --manifest {EVAL_MANIFEST} \
    --out data/eval/train001_eval_baseline.json {' '.join(LIMIT_ARGS)}

# Fine-tuned
!python -X utf8 scripts/eval_asr_phowhisper.py \
    --model models/asr_phowhisper \
    --manifest {EVAL_MANIFEST} \
    --out data/eval/train001_eval_results.json {' '.join(LIMIT_ARGS)}

In [ ]:
import json

with open("data/eval/train001_eval_baseline.json", encoding="utf-8") as f:
    baseline = json.load(f)
with open("data/eval/train001_eval_results.json", encoding="utf-8") as f:
    finetuned = json.load(f)

print(f"Baseline  WER mean (n={baseline['n']}): {baseline['wer_mean']}")
print(f"Fine-tuned WER mean (n={finetuned['n']}): {finetuned['wer_mean']}")
print(f"Target: <0.20 (current baseline ALL=0.183, HN=0.292 — see PROJECT_PROGRESS.md)")

## 10. Download checkpoint + cleanup pilot audio

1. Download `models/asr_phowhisper/` (model + processor files only — no audio).
2. Download `data/eval/train001_eval_baseline.json` + `train001_eval_results.json` (results, no audio).
3. **Delete pilot audio** from this runtime, any Drive mount, or uploaded Dataset — required before closing the session (PILOT PHASE EXCEPTION #2).

In [ ]:
# Colab: download checkpoint + results as a zip
# from google.colab import files
# !zip -r asr_phowhisper_checkpoint.zip models/asr_phowhisper data/eval/train001_eval_results.json data/eval/train001_eval_baseline.json
# files.download("asr_phowhisper_checkpoint.zip")

# Mandatory cleanup — pilot audio (PII) must not remain in the runtime/Drive/Dataset
!rm -rf data/audio/pilot/*
print("Pilot audio cleaned up.")